<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 8 · Predictive Modelling: OLS, Machine and Deep Learning

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook focuses on the OLS baseline that anchors Chapter 8.

- Build a supervised-learning data set of lagged SPY features.
- Fit an OLS model and inspect in-sample and out-of-sample statistics.
- Sketch how predictions map into trading positions.


In [ ]:
import sys
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/yhilpisch/paatcode.git"
    ROOT_DIR = Path("/content/paatcode")
    if not ROOT_DIR.exists():
        subprocess.run([
            "git",
            "clone",
            REPO_URL,
            str(ROOT_DIR),
        ], check=True)
    PROJECT_ROOT = ROOT_DIR
else:
    PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch08_ols_baseline as ch08  # OLS helpers and backtest utilities


### 1. Prepare the Data Set

We reuse `prepare_ml_dataset` to construct lagged features and daily log-returns for SPY, then standardise the features before splitting into train and test windows.

In [ ]:
features, targets = ch08.prepare_ml_dataset(n_lags=3, symbol="SPY")
features_scaled = ch08.standardise_features(features)

n_obs = features_scaled.shape[0]
split_idx = int(0.7 * n_obs)

X_train = features_scaled.iloc[:split_idx].to_numpy()
y_train = targets.iloc[:split_idx].to_numpy()
X_test = features_scaled.iloc[split_idx:].to_numpy()
y_test = targets.iloc[split_idx:].to_numpy()

X_train.shape, X_test.shape


### 2. Fit OLS and Inspect R²

OLS is small enough that we can implement it directly with NumPy linear algebra and compute classical `R²` diagnostics by hand.

In [ ]:
beta = ch08.fit_ols(X_train, y_train)
y_hat_train = ch08.predict_ols(beta, X_train)
y_hat_test = ch08.predict_ols(beta, X_test)

r2_train = ch08.r2_score(y_train, y_hat_train)
r2_test = ch08.r2_score(y_test, y_hat_test)

print(f"R^2 train: {r2_train: .4f}")
print(f"R^2 test : {r2_test: .4f}")


### 3. Visualising Predictions vs. Realised Returns

The scatter plot below uses the test window to show how predicted and realised log-returns align. Later models (trees, neural nets) will plug into the same pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(y_hat_test, y_test, alpha=0.4, s=10)
ax.axhline(0.0, color="black", lw=0.8)
ax.axvline(0.0, color="black", lw=0.8)
ax.set_xlabel("predicted log-return")
ax.set_ylabel("realised log-return")
ax.set_title("OLS predictions vs. realised SPY log-returns (test window)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Chapter 8’s OLS baseline is fully reproducible with a handful of NumPy and pandas calls.
- Train/test splits and simple `R²` diagnostics keep the focus on predictive skill rather than story-driven parameter tweaks.
- Once this pipeline is clear, replacing OLS with richer models changes only the feature construction and the estimator, not the surrounding workflow.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">